# Repair Table SYNC METADATA from pipeline YAML

This notebook reads a pipeline config YAML (e.g. from `example_pipelines/pipeline_1.yaml`), loops over its databases, schemas, and tables, and generates **REPAIR TABLE ... SYNC METADATA** statements for each Iceberg table. You can then run these statements to sync table metadata.

In [ ]:
dbutils.widgets.text("pipeline_yaml_path", "example_pipelines/pipeline_1.yaml", "Pipeline YAML path")
pipeline_yaml_path = dbutils.widgets.get("pipeline_yaml_path")

import yaml

try:
    with open(pipeline_yaml_path, "r") as f:
        config = yaml.safe_load(f) or {}
except FileNotFoundError:
    raise FileNotFoundError(
        f"Pipeline YAML not found at {pipeline_yaml_path}. "
        "Ensure the path is accessible from this cluster (e.g. workspace repo or volume)."
    )

databases = config.get("databases") or {}
print(f"Loaded config: {len(databases)} database(s) from {pipeline_yaml_path}")

In [ ]:
repair_statements = []

for db_key, db_config in databases.items():
    uc_catalog = db_config.get("uc_catalog")
    if not uc_catalog:
        print(f"Warning: skipping database '{db_key}' (missing uc_catalog)")
        continue
    schemas_config = db_config.get("schemas") or {}
    for schema_key, schema_config in schemas_config.items():
        uc_schema = schema_config.get("uc_schema")
        if not uc_schema:
            print(f"Warning: skipping schema '{db_key}.{schema_key}' (missing uc_schema)")
            continue
        tables = schema_config.get("tables") or {}
        for table_name in tables:
            stmt = f"REPAIR TABLE `{uc_catalog}`.`{uc_schema}`.`{table_name}` SYNC METADATA;"
            repair_statements.append(stmt)

print(f"Generated {len(repair_statements)} REPAIR TABLE statement(s).")

In [ ]:
dbutils.widgets.dropdown("run_repair", "No", ["Yes", "No"], "Run REPAIR statements?")
run_repair = dbutils.widgets.get("run_repair") == "Yes"

for stmt in repair_statements:
    print(stmt)

if run_repair and repair_statements:
    print("\n--- Executing REPAIR statements ---")
    for stmt in repair_statements:
        try:
            spark.sql(stmt)
            print(f"OK: {stmt}")
        except Exception as e:
            print(f"FAILED: {stmt}")
            print(f"  Error: {e}")
elif run_repair and not repair_statements:
    print("No statements to run.")
else:
    print("\n(Set 'Run REPAIR statements?' to Yes to execute the statements above.)")